# Unit 2: 神经网络基础

## 学习目标
- 理解 `nn.Module` 的用法和设计理念
- 掌握线性层、激活函数、损失函数
- 学会使用优化器（SGD、Adam）
- 编写标准的 PyTorch 训练循环
- 使用 DataLoader 高效加载数据

## 2.1 nn.Module: 神经网络的基石

`nn.Module` 是所有神经网络模块的基类。你需要：
1. 在 `__init__` 中定义网络层
2. 在 `forward` 中定义前向传播逻辑

PyTorch 会自动追踪 `__init__` 中注册的 `nn.Module` 子模块。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

In [ ]:
# 第一个 nn.Module: 简单线性回归
class LinearRegression(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, output_dim)

    def forward(self, x):
        return self.linear(x)

model = LinearRegression(1, 1)
print(model)
print(f"\n参数列表:")
for name, param in model.named_parameters():
    print(f"  {name}: shape={param.shape}, requires_grad={param.requires_grad}")

## 2.2 激活函数

激活函数引入非线性，让网络能够学习复杂映射。

| 函数 | 公式 | 特点 |
|------|------|------|
| **ReLU** | $\max(0, x)$ | 计算快，缓解梯度消失 |
| **Sigmoid** | $\frac{1}{1+e^{-x}}$ | 输出 (0,1)，易饱和 |
| **Tanh** | $\frac{e^x-e^{-x}}{e^x+e^{-x}}$ | 输出 (-1,1)，零中心 |
| **LeakyReLU** | $\max(0.01x, x)$ | 解决 ReLU 死亡问题 |
| **GELU** | 高斯误差线性单元 | Transformer 常用 |

In [ ]:
# 可视化各种激活函数
x = torch.linspace(-5, 5, 200)

activations = {
    "ReLU": nn.ReLU(),
    "Sigmoid": nn.Sigmoid(),
    "Tanh": nn.Tanh(),
    "LeakyReLU(0.1)": nn.LeakyReLU(0.1),
    "GELU": nn.GELU(),
}

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, (name, act) in enumerate(activations.items()):
    y = act(x)
    axes[i].plot(x.numpy(), y.numpy(), linewidth=2)
    axes[i].set_title(name, fontsize=14)
    axes[i].grid(True, alpha=0.3)
    axes[i].axhline(y=0, color='k', linewidth=0.5)
    axes[i].axvline(x=0, color='k', linewidth=0.5)

axes[-1].axis('off')
plt.tight_layout()
plt.show()

## 2.3 损失函数

损失函数衡量模型预测与真实值的差距。

- **MSELoss**: 回归任务，预测连续值
- **CrossEntropyLoss**: 分类任务（内置 Softmax，输入 logits）
- **BCEWithLogitsLoss**: 二分类（内置 Sigmoid，数值更稳定）

In [ ]:
# 演示 CrossEntropyLoss
# 假设 3 分类问题，batch_size=2
logits = torch.tensor([[0.5, 1.2, -0.3], [2.0, 0.1, 0.8]])  # 未归一化的输出
targets = torch.tensor([1, 0])  # 真实类别

criterion = nn.CrossEntropyLoss()
loss = criterion(logits, targets)
print(f"Logits:\n{logits}")
print(f"Targets: {targets}")
print(f"CrossEntropyLoss: {loss.item():.4f}")

# 手动计算验证: -log(softmax(logits)) 在正确类别处的值
probs = torch.softmax(logits, dim=1)
print(f"Softmax 概率:\n{probs}")
print(f"正确类别概率: sample0={probs[0,1]:.4f}, sample1={probs[1,0]:.4f}")
manual_loss = -(torch.log(probs[0, 1]) + torch.log(probs[1, 0])) / 2
print(f"手动计算: {manual_loss:.4f}")

## 2.4 优化器

优化器根据梯度更新模型参数。

- **SGD**: $\theta_{t+1} = \theta_t - \eta \nabla L$，可加 momentum
- **Adam**: 自适应学习率，最常用，结合了 Momentum + RMSProp
- **AdamW**: Adam + 解耦的权重衰减（推荐用于 Transformer）

In [ ]:
# 优化器使用示范
model = nn.Linear(5, 1)

# SGD with momentum
optimizer_sgd = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# Adam
optimizer_adam = optim.Adam(model.parameters(), lr=0.001)

print("optimizer_sgd:", optimizer_sgd)
print("\noptimizer_adam:", optimizer_adam)

# 优化器核心操作
x = torch.randn(1, 5)
target = torch.randn(1, 1)

optimizer_adam.zero_grad()   # 1. 清零梯度
output = model(x)            # 2. 前向传播
loss = nn.MSELoss()(output, target)  # 3. 计算损失
loss.backward()              # 4. 反向传播
optimizer_adam.step()        # 5. 更新参数

print(f"损失: {loss.item():.4f}")

## 2.5 完整训练循环

标准的 PyTorch 训练/验证循环包含：训练模式切换、batch 迭代、loss 计算、梯度更新、定期验证。

In [ ]:
# 生成合成回归数据
torch.manual_seed(42)
N = 1000
X = torch.randn(N, 3)                    # 3个特征
true_w = torch.tensor([2.0, -1.5, 0.5])  # 真实权重
true_b = torch.tensor(0.3)
y = X @ true_w + true_b + 0.1 * torch.randn(N)  # 加噪声

# 数据集分割
train_size = int(0.8 * N)
X_train, X_val = X[:train_size], X[train_size:]
y_train, y_val = y[:train_size], y[train_size:]

print(f"Train: {X_train.shape}, Val: {X_val.shape}")

In [ ]:
# 构建模型
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)  # 去掉最后一维

model = MLP(3, 64, 1).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)
print(model)

In [ ]:
# 标准训练循环
epochs = 200
train_losses = []
val_losses = []

for epoch in range(epochs):
    # ---- 训练阶段 ----
    model.train()
    optimizer.zero_grad()
    pred = model(X_train.to(device))
    loss = criterion(pred, y_train.to(device))
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())

    # ---- 验证阶段 ----
    model.eval()
    with torch.no_grad():
        val_pred = model(X_val.to(device))
        val_loss = criterion(val_pred, y_val.to(device))
        val_losses.append(val_loss.item())

    if (epoch + 1) % 40 == 0:
        print(f"Epoch {epoch+1:3d} | Train Loss: {loss.item():.4f} | Val Loss: {val_loss.item():.4f}")

In [ ]:
# 可视化训练曲线
plt.figure(figsize=(10, 4))
plt.plot(train_losses, label="Train Loss", alpha=0.7)
plt.plot(val_losses, label="Val Loss", alpha=0.7)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 检查学到的参数
print(f"True weights: {true_w.tolist()}, bias: {true_b.item():.3f}")
print(f"Final Val Loss: {val_losses[-1]:.6f}")

## 2.6 DataLoader 和 Dataset

`Dataset` 定义了如何获取单个样本，`DataLoader` 提供批量加载、打乱、多线程等功能。

In [ ]:
# 使用 DataLoader 进行批量训练
batch_size = 64
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

val_dataset = TensorDataset(X_val, y_val)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Batch 数量: train={len(train_loader)}, val={len(val_loader)}")

# 查看一个 batch
for batch_x, batch_y in train_loader:
    print(f"batch_x shape: {batch_x.shape}, batch_y shape: {batch_y.shape}")
    break

In [ ]:
# 使用 DataLoader 的完整训练循环
model2 = MLP(3, 64, 1).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model2.parameters(), lr=0.01)

for epoch in range(100):
    # 训练
    model2.train()
    total_train_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        pred = model2(batch_x)
        loss = criterion(pred, batch_y)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()

    # 验证
    model2.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            pred = model2(batch_x)
            total_val_loss += criterion(pred, batch_y).item()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d} | Train Loss: {total_train_loss/len(train_loader):.4f} | Val Loss: {total_val_loss/len(val_loader):.4f}")

## 2.7 模型保存与加载

推荐保存 `state_dict`（只存参数），而不是整个模型对象。

In [ ]:
# 保存模型
torch.save(model.state_dict(), "/tmp/model_demo.pth")
print("模型已保存")

# 加载模型
loaded_model = MLP(3, 64, 1).to(device)
loaded_model.load_state_dict(torch.load("/tmp/model_demo.pth", map_location=device, weights_only=True))
loaded_model.eval()
print("模型已加载")

# 验证加载后预测一致
with torch.no_grad():
    original_pred = model(X_val.to(device))
    loaded_pred = loaded_model(X_val.to(device))
    diff = (original_pred - loaded_pred).abs().max().item()
    print(f"最大预测差异: {diff:.10f}")

## 要点总结

| 概念 | 关键点 |
|------|--------|
| nn.Module | `__init__` 定义层，`forward` 定义计算 |
| 激活函数 | 引入非线性; ReLU 最常用，注意死神经元 |
| 损失函数 | 回归用 MSE，分类用 CrossEntropyLoss |
| 优化器 | Adam 自适应学习率，SGD+Momentum 泛化好 |
| 训练循环 | 5 步: `zero_grad → forward → loss → backward → step` |
| DataLoader | 自动批量、打乱、多线程加载 |
| 保存加载 | 使用 `state_dict()`，不要直接保存整个模型 |

### 练习
1. 修改 MLP 隐藏层大小和层数，观察对训练的影响
2. 尝试用 SGD 替代 Adam，调整学习率使模型收敛
3. 实现 `nn.Dropout` 正则化，观察验证集表现